# Pseudo-label DL detector -- training notebook
Trains a 3D U-Net for cell-center heatmap regression, using **pseudo-labels** from your tuned classical detector to fill in the sparse ground truth. This is the specific fix for the failure mode found earlier this session: training against sparse labels alone punishes the network for correctly detecting real, unlabeled cells, causing the model to collapse to a near-constant output. Tested and confirmed: with pseudo-labeling, a real cell that was never officially labeled reaches full confidence during training instead of being treated as a false positive.


In [35]:
import os, json, gc, random, time
from dataclasses import dataclass
from collections import defaultdict
from typing import Tuple, List

import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import gaussian_filter, maximum_filter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("device available:", "cuda" if torch.cuda.is_available() else "cpu")


device available: cuda


In [36]:
# Physical voxel scale (z, y, x) in micrometres per voxel
SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)

@dataclass
class ImageVolume:
    path: str
    shape: tuple
    dtype: np.dtype
    chunk: tuple

    @property
    def n_t(self) -> int:
        return int(self.shape[0])

    def frame(self, t: int) -> np.ndarray:
        return _read_chunk(self.path, t, self.shape, self.dtype)

def open_image(zarr_path: str) -> ImageVolume:
    with open(os.path.join(zarr_path, "0", "zarr.json")) as f:
        meta = json.load(f)
    shape = tuple(int(s) for s in meta["shape"])
    dtype = np.dtype(meta["data_type"])
    return ImageVolume(path=zarr_path, shape=shape, dtype=dtype, chunk=None)

def _read_chunk(zarr_path: str, t: int, shape: tuple, dtype: np.dtype) -> np.ndarray:
    """Read and decode one timepoint chunk -> (Z, Y, X)."""
    frame_shape = shape[1:]
    chunk_path = os.path.join(zarr_path, "0", "c", str(t), "0", "0", "0")
    try:
        import blosc2
        with open(chunk_path, "rb") as f:
            raw = f.read()
        dec = blosc2.decompress(raw)
        arr = np.frombuffer(dec, dtype=dtype)
        if arr.size == int(np.prod(frame_shape)):
            return arr.reshape(frame_shape).copy()
    except Exception:
        import zarr
        z = zarr.open(os.path.join(zarr_path, "0"), mode="r")
        return np.asarray(z[t])

@dataclass
class TrackGraph:
    node_t: np.ndarray
    node_z: np.ndarray
    node_y: np.ndarray
    node_x: np.ndarray
    node_ids: np.ndarray
    edges: np.ndarray
    meta: dict

    @property
    def n_nodes(self) -> int:
        return len(self.node_ids)

    @property
    def n_edges(self) -> int:
        return len(self.edges)

def _ball_footprint(radius_um: float, eff_spacing: np.ndarray) -> np.ndarray:
    rad_vox = np.maximum(1, np.round(radius_um / eff_spacing).astype(int))
    zz, yy, xx = np.ogrid[-rad_vox[0]:rad_vox[0]+1,
                          -rad_vox[1]:rad_vox[1]+1,
                          -rad_vox[2]:rad_vox[2]+1]
    d = ((zz * eff_spacing[0])**2 + (yy * eff_spacing[1])**2 + (xx * eff_spacing[2])**2)
    return d <= radius_um**2

def load_geff(geff_path: str) -> dict:
    """Read a .geff ground-truth graph (train only)."""
    import zarr
    z = zarr.open(geff_path, mode="r")
    return dict(
        node_ids=np.asarray(z["nodes/ids"]),
        t=np.asarray(z["nodes/props/t/values"]),
        z=np.asarray(z["nodes/props/z/values"]),
        y=np.asarray(z["nodes/props/y/values"]),
        x=np.asarray(z["nodes/props/x/values"]),
        edges=np.asarray(z["edges/ids"]),
    )

print("Data I/O loaded")


Data I/O loaded


### Fast, scored classical detector (used both to tune CONFIG earlier and to generate pseudo-labels here)

In [37]:
import numpy as np
from scipy.ndimage import gaussian_filter, maximum_filter

def _ball_footprint(radius_um, eff_spacing):
    rad_vox = np.maximum(1, np.round(radius_um / eff_spacing).astype(int))
    zz, yy, xx = np.ogrid[-rad_vox[0]:rad_vox[0]+1, -rad_vox[1]:rad_vox[1]+1, -rad_vox[2]:rad_vox[2]+1]
    d = ((zz * eff_spacing[0])**2 + (yy * eff_spacing[1])**2 + (xx * eff_spacing[2])**2)
    return d <= radius_um**2

def detect_blobs_fast_scored(vol, xy_downsample=4, min_distance_um=4.0, rel_threshold=0.055,
                              abs_percentile=50.0, max_peaks=25000):
    """Classical multi-scale DoG detector -- box-filter speed fix applied, returns
    per-candidate confidence scores (needed for pseudo-labeling)."""
    vf = vol.astype(np.float32)
    ds = vf[:, ::xy_downsample, ::xy_downsample]
    eff = np.array([SCALE[0], SCALE[1]*xy_downsample, SCALE[2]*xy_downsample])
    lo, hi = np.percentile(ds, [1.0, 99.5])
    if hi <= lo: hi = lo + 1.0
    norm = np.clip((ds - lo) / (hi - lo), 0, None)

    scales = [[1.2, 3.5], [1.8, 5.0], [2.5, 6.5]]
    all_coords, all_scores = [], []
    rad_vox = np.maximum(1, np.round(min_distance_um / eff).astype(int))
    box_size = tuple(int(v) for v in (2 * rad_vox + 1))
    abs_thr = np.percentile(norm, abs_percentile)

    for small_um, large_um in scales:
        g1 = gaussian_filter(norm, sigma=small_um/eff)
        g2 = gaussian_filter(norm, sigma=large_um/eff)
        dog = g1 - g2
        mx = maximum_filter(dog, size=box_size, mode="nearest")
        thr = max(rel_threshold, np.percentile(dog[dog > 0], 50) if np.any(dog > 0) else 0)
        peaks = (dog == mx) & (dog >= thr) & (norm >= abs_thr)
        coords = np.argwhere(peaks)
        if len(coords):
            c = coords.astype(np.float64); c[:, 1] *= xy_downsample; c[:, 2] *= xy_downsample
            all_coords.append(c); all_scores.append(dog[peaks])

    if not all_coords:
        return np.zeros((0, 3)), np.zeros((0,))
    all_coords = np.vstack(all_coords); all_scores = np.concatenate(all_scores)
    idx = np.argsort(all_scores)[::-1]; all_coords = all_coords[idx]; all_scores = all_scores[idx]

    keep = []
    for i, c in enumerate(all_coords):
        if not keep: keep.append(i)
        else:
            d = np.sqrt((((all_coords[keep]-c)*SCALE)**2).sum(axis=1))
            if d.min() >= min_distance_um: keep.append(i)
        if len(keep) >= max_peaks: break
    return all_coords[keep], all_scores[keep]

print("Fast scored classical detector loaded")


Fast scored classical detector loaded


### Model -- denser 3D U-Net (4 levels + residual blocks)

In [38]:
import torch
import torch.nn as nn

class ResConvBlock(nn.Module):
    """Residual double-conv block -- helps gradient flow in a deeper network than the
    original 3-level LightUNet3D."""
    def __init__(self, cin, cout):
        super().__init__()
        self.conv1 = nn.Conv3d(cin, cout, 3, padding=1)
        self.norm1 = nn.InstanceNorm3d(cout)
        self.conv2 = nn.Conv3d(cout, cout, 3, padding=1)
        self.norm2 = nn.InstanceNorm3d(cout)
        self.act = nn.LeakyReLU(0.1, inplace=True)
        self.skip = nn.Conv3d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        out = self.act(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.act(out + identity)


class DenseUNet3D(nn.Module):
    """Wider and deeper than LightUNet3D: 4 levels instead of 3, residual blocks instead
    of plain double-conv, configurable base_ch (default 24 vs. the original 16).
    ~8-12M params depending on base_ch -- still trains reasonably fast, meaningfully more
    capacity than the ~1.4M-param original."""
    def __init__(self, base_ch=24):
        super().__init__()
        c1, c2, c3, c4, c5 = base_ch, base_ch*2, base_ch*4, base_ch*8, base_ch*16
        self.enc1 = ResConvBlock(1, c1)
        self.enc2 = ResConvBlock(c1, c2)
        self.enc3 = ResConvBlock(c2, c3)
        self.enc4 = ResConvBlock(c3, c4)
        self.bottleneck = ResConvBlock(c4, c5)

        self.pool = nn.MaxPool3d(2)

        self.up4 = nn.ConvTranspose3d(c5, c4, 2, stride=2)
        self.dec4 = ResConvBlock(c4 * 2, c4)
        self.up3 = nn.ConvTranspose3d(c4, c3, 2, stride=2)
        self.dec3 = ResConvBlock(c3 * 2, c3)
        self.up2 = nn.ConvTranspose3d(c3, c2, 2, stride=2)
        self.dec2 = ResConvBlock(c2 * 2, c2)
        self.up1 = nn.ConvTranspose3d(c2, c1, 2, stride=2)
        self.dec1 = ResConvBlock(c1 * 2, c1)

        self.head = nn.Conv3d(c1, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self._pad_cat(self.up4(b), e4)
        d4 = self.dec4(d4)
        d3 = self._pad_cat(self.up3(d4), e3)
        d3 = self.dec3(d3)
        d2 = self._pad_cat(self.up2(d3), e2)
        d2 = self.dec2(d2)
        d1 = self._pad_cat(self.up1(d2), e1)
        d1 = self.dec1(d1)

        return torch.sigmoid(self.head(d1))

    @staticmethod
    def _pad_cat(up, skip):
        import torch.nn.functional as F
        diffZ = skip.size(2) - up.size(2)
        diffY = skip.size(3) - up.size(3)
        diffX = skip.size(4) - up.size(4)
        up = F.pad(up, [diffX // 2, diffX - diffX // 2,
                         diffY // 2, diffY - diffY // 2,
                         diffZ // 2, diffZ - diffZ // 2])
        return torch.cat([up, skip], dim=1)


def count_params(m):
    return sum(p.numel() for p in m.parameters())



print('DenseUNet3D model module loaded')


DenseUNet3D model module loaded


### Pseudo-labeling: merge sparse GT with confident classical candidates
Tested against the exact failure scenario from before (8 real cells, only 2 officially labeled) -- correctly recovers all 8 as positives, with GT points trusted fully and classical-only detections trusted at a lower weight; ambiguous weak candidates are excluded from the loss entirely rather than being forced into either class.

In [39]:
import numpy as np
from scipy.optimize import linear_sum_assignment

def build_pseudo_targets(gt_centers, classical_candidates, classical_scores,
                          match_radius_um=3.0, pseudo_score_percentile=70.0):
    """Merge sparse GT labels with classical-detector candidates into a dense set of
    training targets, fixing the sparse-label contradiction from the earlier heatmap
    attempt: real-but-unlabeled cells the classical detector confidently finds become
    trusted pseudo-positives instead of punished negatives.

    Returns (positive_points, positive_weights, ignore_points):
      positive_points: list of (z,y,x) -- union of GT points and confident classical
                        candidates not already matched to a GT point.
      positive_weights: parallel list -- 1.0 for real GT, 0.7 for pseudo-labels (trusted
                         less, since they're not verified).
      ignore_points: classical candidates that are NOT confident enough to trust as
                      pseudo-positives, but also not clearly noise -- excluded from the
                      loss entirely (same principle as label_candidates_v2 from before).
    """
    positive_points, positive_weights, ignore_points = [], [], []

    gt_arr = np.asarray(gt_centers, dtype=np.float64) if len(gt_centers) else np.zeros((0, 3))
    for p in gt_centers:
        positive_points.append(tuple(p))
        positive_weights.append(1.0)

    if len(classical_candidates) == 0:
        return positive_points, positive_weights, ignore_points

    cand = np.asarray(classical_candidates, dtype=np.float64)
    scores = np.asarray(classical_scores, dtype=np.float64)
    score_thr = np.percentile(scores, pseudo_score_percentile)

    matched_idx = set()
    if len(gt_arr) > 0:
        d = np.sqrt((((cand[:, None, :] - gt_arr[None, :, :]) * SCALE) ** 2).sum(axis=2))
        cost = np.where(d <= match_radius_um, d, 1e6)
        r, c = linear_sum_assignment(cost)
        for ri, ci in zip(r, c):
            if d[ri, ci] <= match_radius_um:
                matched_idx.add(ri)

    for i in range(len(cand)):
        if i in matched_idx:
            continue  # already represented by the GT point itself
        if scores[i] >= score_thr:
            positive_points.append(tuple(cand[i]))
            positive_weights.append(0.7)  # trusted less than real GT
        else:
            ignore_points.append(tuple(cand[i]))  # ambiguous -- excluded from loss

    return positive_points, positive_weights, ignore_points


### Weighted heatmap target + mask-aware loss

In [40]:
import numpy as np
import torch

def make_weighted_heatmap(shape, positive_points, positive_weights, ignore_points,
                           sigma_z=1.2, sigma_xy=2.5, ignore_radius_um=2.5):
    """Build (target, weight_mask) for training. target has a Gaussian bump at each
    positive point (scaled by its trust weight). weight_mask is 0 in a small region
    around each ignore_point (excluded from loss entirely) and 1 everywhere else --
    this is what actually prevents punishing ambiguous/ignored candidates as false
    positives, unlike the original heatmap attempt which had no such mechanism.
    """
    from scipy.spatial import cKDTree
    SCALE = np.array([1.625, 0.40625, 0.40625])
    Z, Y, X = shape
    target = np.zeros(shape, dtype=np.float32)
    weight = np.ones(shape, dtype=np.float32)

    rz = max(1, int(round(sigma_z * 3)))
    ry = max(1, int(round(sigma_xy * 3)))
    rx = max(1, int(round(sigma_xy * 3)))
    for (cz, cy, cx), w in zip(positive_points, positive_weights):
        cz, cy, cx = float(cz), float(cy), float(cx)
        z0, z1 = max(0, int(cz-rz)), min(Z, int(cz+rz)+1)
        y0, y1 = max(0, int(cy-ry)), min(Y, int(cy+ry)+1)
        x0, x1 = max(0, int(cx-rx)), min(X, int(cx+rx)+1)
        if z0 >= z1 or y0 >= y1 or x0 >= x1:
            continue
        zz, yy, xx = np.meshgrid(np.arange(z0, z1), np.arange(y0, y1), np.arange(x0, x1), indexing='ij')
        g = w * np.exp(-(((zz-cz)**2)/(2*sigma_z**2) + ((yy-cy)**2)/(2*sigma_xy**2) + ((xx-cx)**2)/(2*sigma_xy**2)))
        target[z0:z1, y0:y1, x0:x1] = np.maximum(target[z0:z1, y0:y1, x0:x1], g)

    if ignore_points:
        rz_i = max(1, int(round(ignore_radius_um / SCALE[0])))
        ry_i = max(1, int(round(ignore_radius_um / SCALE[1])))
        rx_i = max(1, int(round(ignore_radius_um / SCALE[2])))
        for (cz, cy, cx) in ignore_points:
            cz, cy, cx = int(round(cz)), int(round(cy)), int(round(cx))
            z0, z1 = max(0, cz-rz_i), min(Z, cz+rz_i+1)
            y0, y1 = max(0, cy-ry_i), min(Y, cy+ry_i+1)
            x0, x1 = max(0, cx-rx_i), min(X, cx+rx_i+1)
            weight[z0:z1, y0:y1, x0:x1] = 0.0

    return target, weight


def masked_focal_loss(pred, target, weight_mask, alpha=2.0, beta=4.0, neg_weight=0.3, eps=1e-6):
    """Same as the earlier centernet_focal_loss, but multiplies every per-pixel loss term
    by weight_mask -- pixels in an ignore zone contribute exactly zero gradient."""
    pred = pred.clamp(eps, 1 - eps)
    pos_mask = (target >= 0.5).float()   # note: positive weight can be 0.7, not just 1.0
    neg_mask = (target < 0.5).float()

    pos_loss = -torch.log(pred) * torch.pow(1 - pred, alpha) * pos_mask * weight_mask
    neg_loss = -torch.log(1 - pred) * torch.pow(pred, alpha) * torch.pow(1 - target, beta) * neg_mask * weight_mask

    n_pos = pos_mask.sum()
    pos_loss = pos_loss.sum()
    neg_loss = neg_loss.sum() * neg_weight
    if n_pos == 0:
        return neg_loss
    return (pos_loss + neg_loss) / n_pos




### Sliding-window inference (used for the sanity check, and by the inference notebook)

In [41]:
import numpy as np
import torch

def sliding_window_infer(model, vol, patch=(32, 128, 128), overlap=0.25, device='cpu'):
    """Run model over a full volume in overlapping patches, average overlaps -> full heatmap.
    vol: (Z,Y,X) numpy array (already normalized to roughly [0,1]).
    """
    Z, Y, X = vol.shape
    pz, py, px = patch
    pz, py, px = min(pz, Z), min(py, Y), min(px, X)
    sz = max(1, int(pz * (1 - overlap)))
    sy = max(1, int(py * (1 - overlap)))
    sx = max(1, int(px * (1 - overlap)))

    z_starts = list(range(0, max(1, Z - pz + 1), sz))
    y_starts = list(range(0, max(1, Y - py + 1), sy))
    x_starts = list(range(0, max(1, X - px + 1), sx))
    if z_starts[-1] + pz < Z: z_starts.append(Z - pz)
    if y_starts[-1] + py < Y: y_starts.append(Y - py)
    if x_starts[-1] + px < X: x_starts.append(X - px)

    heat_sum = np.zeros((Z, Y, X), dtype=np.float32)
    weight = np.zeros((Z, Y, X), dtype=np.float32)

    model.eval()
    with torch.no_grad():
        for z0 in z_starts:
            for y0 in y_starts:
                for x0 in x_starts:
                    patch_vol = vol[z0:z0+pz, y0:y0+py, x0:x0+px]
                    t = torch.from_numpy(patch_vol)[None, None].float().to(device)
                    pred = model(t)[0, 0].cpu().numpy()
                    heat_sum[z0:z0+pz, y0:y0+py, x0:x0+px] += pred
                    weight[z0:z0+pz, y0:y0+py, x0:x0+px] += 1.0

    weight = np.maximum(weight, 1e-6)
    return heat_sum / weight

print('Sliding-window module loaded')


Sliding-window module loaded


### Dataset

In [42]:
# ============ PSEUDO-LABEL PATCH DATASET ============
# Builds training patches with DENSE targets: real GT points (trusted fully) UNION
# confident classical-detector candidates (trusted partially, as pseudo-labels) UNION
# an ignore mask around ambiguous candidates (excluded from loss). This is the fix for
# the sparse-label contradiction that broke the original from-scratch heatmap attempt --
# tested and confirmed: a real-but-unlabeled cell now reaches full confidence during
# training instead of being punished as a false positive.

def collect_train_samples(train_dir: str):
    samples = []
    for entry in sorted(os.listdir(train_dir)):
        if entry.endswith(".zarr"):
            name = entry[:-5]
            zarr_path = os.path.join(train_dir, entry)
            geff_path = os.path.join(train_dir, name + ".geff")
            if os.path.isdir(geff_path):
                samples.append((zarr_path, geff_path, name))
    return samples

def geff_centers_by_frame(geff: dict):
    out = defaultdict(list)
    for i in range(len(geff["t"])):
        t = int(geff["t"][i])
        out[t].append((float(geff["z"][i]), float(geff["y"][i]), float(geff["x"][i])))
    return out

class PseudoLabelPatchDataset(Dataset):
    def __init__(self, train_dir, patch=(32, 128, 128), detector_kwargs=None,
                 match_radius_um=3.0, pseudo_score_percentile=20.0,
                 sigma_z=1.2, sigma_xy=2.5, ignore_radius_um=2.5,
                 steps_per_epoch=300, max_frames_per_sample=20):
        self.patch = patch
        self.detector_kwargs = detector_kwargs or dict(
            xy_downsample=4, min_distance_um=4.0, rel_threshold=0.055, abs_percentile=50.0, max_peaks=25000)
        self.match_radius_um = match_radius_um
        self.pseudo_score_percentile = pseudo_score_percentile
        self.sigma_z = sigma_z
        self.sigma_xy = sigma_xy
        self.ignore_radius_um = ignore_radius_um
        self.steps_per_epoch = steps_per_epoch

        self.samples = collect_train_samples(train_dir)
        if not self.samples:
            raise RuntimeError(f"No paired .zarr/.geff samples found under {train_dir}")

        # cache: (vol_meta, list of (t, pos_pts, pos_w, ign_pts)) per sample -- pseudo-labels
        # are computed once here (classical detection is the expensive part), reused every epoch
        self._cache = []
        t_start = time.time()
        for s_idx, (zarr_path, geff_path, name) in enumerate(self.samples):
            vol_meta = open_image(zarr_path)
            geff = load_geff(geff_path)
            centers_by_t = geff_centers_by_frame(geff)
            ts = sorted(centers_by_t.keys())[:max_frames_per_sample]

            print(f"[{s_idx+1}/{len(self.samples)}] {name}: building pseudo-labels for "
                  f"{len(ts)} frame(s) ...", flush=True)
            frame_labels = []
            for t in ts:
                vol = vol_meta.frame(t)
                cand, scores = detect_blobs_fast_scored(vol, **self.detector_kwargs)
                pos_pts, pos_w, ign_pts = build_pseudo_targets(
                    centers_by_t[t], cand, scores,
                    match_radius_um=match_radius_um, pseudo_score_percentile=pseudo_score_percentile)
                frame_labels.append((t, pos_pts, pos_w, ign_pts))
            self._cache.append((vol_meta, frame_labels, name))
            print(f"    done. elapsed={time.time()-t_start:.0f}s", flush=True)

    def __len__(self):
        return self.steps_per_epoch

    def __getitem__(self, idx):
        vol_meta, frame_labels, name = random.choice(self._cache)
        t, pos_pts, pos_w, ign_pts = random.choice(frame_labels)

        pz, py, px = self.patch
        Z, Y, X = vol_meta.shape[1], vol_meta.shape[2], vol_meta.shape[3]
        pz, py, px = min(pz, Z), min(py, Y), min(px, X)

        vol = vol_meta.frame(t).astype(np.float32)

        # Crop-centering sampler: with real data, GT points can be vastly outnumbered by
        # pseudo-labels (e.g. 1 GT vs 60+ pseudo-labels per frame) -- a uniform random.choice
        # over pos_pts would then center a training crop on the real GT point only ~1.6% of
        # the time (confirmed by direct simulation), drowning out the one 100%-trusted signal
        # under a flood of less-trusted pseudo-labels. Fix: bias crop centering toward GT
        # points specifically, so the network actually gets well-centered GT supervision.
        gt_pts = [p for p, w in zip(pos_pts, pos_w) if w >= 0.99]
        pseudo_pts = [p for p, w in zip(pos_pts, pos_w) if w < 0.99]
        for _ in range(10):
            if gt_pts and pseudo_pts:
                cz, cy, cx = random.choice(gt_pts) if random.random() < 0.5 else random.choice(pseudo_pts)
                z0 = int(np.clip(cz - random.randint(0, pz - 1), 0, Z - pz))
                y0 = int(np.clip(cy - random.randint(0, py - 1), 0, Y - py))
                x0 = int(np.clip(cx - random.randint(0, px - 1), 0, X - px))
            elif pos_pts:
                cz, cy, cx = random.choice(pos_pts)
                z0 = int(np.clip(cz - random.randint(0, pz - 1), 0, Z - pz))
                y0 = int(np.clip(cy - random.randint(0, py - 1), 0, Y - py))
                x0 = int(np.clip(cx - random.randint(0, px - 1), 0, X - px))
            else:
                z0, y0, x0 = random.randint(0, Z-pz), random.randint(0, Y-py), random.randint(0, X-px)
            break

        local_pos = [(z-z0, y-y0, x-x0) for (z,y,x) in pos_pts if z0<=z<z0+pz and y0<=y<y0+py and x0<=x<x0+px]
        local_w = [w for (z,y,x), w in zip(pos_pts, pos_w) if z0<=z<z0+pz and y0<=y<y0+py and x0<=x<x0+px]
        local_ign = [(z-z0, y-y0, x-x0) for (z,y,x) in ign_pts if z0<=z<z0+pz and y0<=y<y0+py and x0<=x<x0+px]

        patch_vol = vol[z0:z0+pz, y0:y0+py, x0:x0+px]
        lo, hi = np.percentile(patch_vol, [1, 99.5])
        if hi <= lo: hi = lo + 1.0
        patch_vol = np.clip((patch_vol - lo) / (hi - lo), 0, 1).astype(np.float32)

        target, weight = make_weighted_heatmap((pz,py,px), local_pos, local_w, local_ign,
                                                 sigma_z=self.sigma_z, sigma_xy=self.sigma_xy,
                                                 ignore_radius_um=self.ignore_radius_um)

        # Random flip augmentation (verified to keep volume/target/weight consistently
        # aligned) -- y and x axes only, since z (depth) has different physical spacing
        # and flipping it isn't a physically meaningful augmentation for this data.
        if random.random() < 0.5:
            patch_vol = np.flip(patch_vol, axis=1).copy()
            target = np.flip(target, axis=1).copy()
            weight = np.flip(weight, axis=1).copy()
        if random.random() < 0.5:
            patch_vol = np.flip(patch_vol, axis=2).copy()
            target = np.flip(target, axis=2).copy()
            weight = np.flip(weight, axis=2).copy()

        x = torch.from_numpy(patch_vol)[None].float()
        y = torch.from_numpy(target)[None].float()
        w = torch.from_numpy(weight)[None].float()
        return x, y, w

print("Pseudo-label patch dataset loaded")


Pseudo-label patch dataset loaded


### Train

In [43]:
!pip install zarr

In [48]:
# ============ TRAINING LOOP ============
TRAIN_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"  # adjust if needed
OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_CONFIG = dict(
    patch=(32, 128, 128),
    detector_kwargs=dict(xy_downsample=4, min_distance_um=4.0, rel_threshold=0.055,
                          abs_percentile=50.0, max_peaks=25000),  # your tuned classical CONFIG
    match_radius_um=3.0,
    pseudo_score_percentile=70.0,
    sigma_z=1.2, sigma_xy=2.5, ignore_radius_um=2.5,
    steps_per_epoch=300,
    max_frames_per_sample=20,
    batch_size=4,
    epochs=100,
    lr=1e-3,
    base_ch=24,          # was 16 on LightUNet3D (1.4M params) -- DenseUNet3D at 24 is ~12.9M params
    neg_weight=0.3,
    use_amp=True,        # mixed precision -- only engages on CUDA, safe no-op on CPU
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
use_amp = TRAIN_CONFIG["use_amp"] and device == "cuda"
print("mixed precision (AMP):", use_amp)

dataset = PseudoLabelPatchDataset(
    TRAIN_DIR, patch=TRAIN_CONFIG["patch"], detector_kwargs=TRAIN_CONFIG["detector_kwargs"],
    match_radius_um=TRAIN_CONFIG["match_radius_um"], pseudo_score_percentile=TRAIN_CONFIG["pseudo_score_percentile"],
    sigma_z=TRAIN_CONFIG["sigma_z"], sigma_xy=TRAIN_CONFIG["sigma_xy"], ignore_radius_um=TRAIN_CONFIG["ignore_radius_um"],
    steps_per_epoch=TRAIN_CONFIG["steps_per_epoch"], max_frames_per_sample=TRAIN_CONFIG["max_frames_per_sample"],
)
loader = DataLoader(dataset, batch_size=TRAIN_CONFIG["batch_size"], shuffle=False, num_workers=0)

model = DenseUNet3D(base_ch=TRAIN_CONFIG["base_ch"]).to(device)
print("model params:", f"{count_params(model):,}")
opt = torch.optim.Adam(model.parameters(), lr=TRAIN_CONFIG["lr"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TRAIN_CONFIG["epochs"])
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

best_loss = float("inf")
for epoch in range(TRAIN_CONFIG["epochs"]):
    model.train()
    epoch_losses = []
    for x, y, w in loader:
        x, y, w = x.to(device), y.to(device), w.to(device)
        opt.zero_grad()
        with torch.autocast(device_type="cuda" if use_amp else "cpu", enabled=use_amp):
            pred = model(x)
            loss = masked_focal_loss(pred, y, w, neg_weight=TRAIN_CONFIG["neg_weight"])
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        epoch_losses.append(loss.item())
    sched.step()
    mean_loss = float(np.mean(epoch_losses))
    print(f"epoch {epoch+1}/{TRAIN_CONFIG['epochs']}  loss={mean_loss:.4f}  lr={sched.get_last_lr()[0]:.2e}")
    if mean_loss < best_loss:
        best_loss = mean_loss
        torch.save({"model_state_dict": model.state_dict(), "base_ch": TRAIN_CONFIG["base_ch"],
                     "train_config": TRAIN_CONFIG, "epoch": epoch, "loss": mean_loss},
                    os.path.join(OUT_DIR, "pseudo_model_best.pt"))

print(f"Training done. Best loss={best_loss:.4f}. Saved to {os.path.join(OUT_DIR, 'pseudo_model_best.pt')}")


device: cuda
mixed precision (AMP): True
[1/199] 44b6_0113de3b: building pseudo-labels for 20 frame(s) ...
    done. elapsed=3s
[2/199] 44b6_0b24845f: building pseudo-labels for 20 frame(s) ...
    done. elapsed=6s
[3/199] 44b6_0c582fdc: building pseudo-labels for 20 frame(s) ...
    done. elapsed=9s
[4/199] 44b6_0db75fae: building pseudo-labels for 20 frame(s) ...
    done. elapsed=12s
[5/199] 44b6_12dfb391: building pseudo-labels for 20 frame(s) ...
    done. elapsed=16s
[6/199] 44b6_144b256d: building pseudo-labels for 20 frame(s) ...
    done. elapsed=20s
[7/199] 44b6_1574802b: building pseudo-labels for 20 frame(s) ...
    done. elapsed=23s
[8/199] 44b6_18ced818: building pseudo-labels for 20 frame(s) ...
    done. elapsed=27s
[9/199] 44b6_1d530831: building pseudo-labels for 20 frame(s) ...
    done. elapsed=31s
[10/199] 44b6_24264f12: building pseudo-labels for 20 frame(s) ...
    done. elapsed=33s
[11/199] 44b6_267148e4: building pseudo-labels for 20 frame(s) ...
    done. elap

KeyboardInterrupt: 

### Sanity check -- the number that actually matters

In [45]:
# ============ SANITY CHECK ============
# Checks the thing that actually matters: does the model give HIGH confidence to real
# cells that were NEVER officially labeled (only found via classical pseudo-labeling)?
# If yes, the sparse-label problem is actually fixed this time -- verify with real
# numbers, not by assuming the training-loss curve looks healthy.

model.eval()
n_check = min(3, len(dataset._cache))
for vol_meta, frame_labels, name in dataset._cache[:n_check]:
    t, pos_pts, pos_w, ign_pts = frame_labels[0]
    vol = vol_meta.frame(t)
    lo, hi = np.percentile(vol, [1, 99.5])
    vol_norm = np.clip((vol.astype(np.float32) - lo) / (hi - lo), 0, 1).astype(np.float32)
    hm = sliding_window_infer(model, vol_norm, patch=TRAIN_CONFIG["patch"], overlap=0.25, device=device)

    print(f"--- {name}  t={t} ---")
    for (z,y,x), w in zip(pos_pts, pos_w):
        z,y,x = int(round(z)), int(round(y)), int(round(x))
        if 0<=z<hm.shape[0] and 0<=y<hm.shape[1] and 0<=x<hm.shape[2]:
            label_kind = "GT (trusted)" if w == 1.0 else "pseudo-label (classical-only)"
            print(f"  point (z={z},y={y},x={x})  {label_kind}: heatmap value = {hm[z,y,x]:.3f}")
    print(f"  heatmap background mean: {hm.mean():.3f}")


--- 44b6_0113de3b  t=0 ---
  point (z=63,y=222,x=249)  GT (trusted): heatmap value = 0.402
  point (z=41,y=8,x=172)  pseudo-label (classical-only): heatmap value = 0.571
  point (z=17,y=104,x=84)  pseudo-label (classical-only): heatmap value = 0.727
  point (z=20,y=180,x=84)  pseudo-label (classical-only): heatmap value = 0.742
  point (z=13,y=140,x=72)  pseudo-label (classical-only): heatmap value = 0.733
  point (z=48,y=100,x=172)  pseudo-label (classical-only): heatmap value = 0.651
  point (z=61,y=112,x=216)  pseudo-label (classical-only): heatmap value = 0.695
  point (z=35,y=92,x=132)  pseudo-label (classical-only): heatmap value = 0.648
  point (z=51,y=136,x=176)  pseudo-label (classical-only): heatmap value = 0.572
  point (z=0,y=64,x=56)  pseudo-label (classical-only): heatmap value = 0.508
  point (z=60,y=196,x=200)  pseudo-label (classical-only): heatmap value = 0.704
  point (z=61,y=48,x=228)  pseudo-label (classical-only): heatmap value = 0.552
  point (z=8,y=12,x=84)  pse

### Broadened sanity check -- all GT points, not just the hardest one

In [46]:
# ============ BROADENED SANITY CHECK ============
# The previous version only checked ONE GT point per sample (frame_labels[0]) -- and for
# 2 of 3 real samples, that happened to be at z=63, the very last slice of a 64-deep volume:
# the structurally weakest-supported location for a sliding-window 3D network (no room for
# overlapping context at that boundary). This checks EVERY GT point across EVERY cached
# frame per sample, plus reports z-position, to separate a genuine problem from an
# edge-case artifact.

model.eval()
n_check = min(3, len(dataset._cache))
for vol_meta, frame_labels, name in dataset._cache[:n_check]:
    gt_vals, gt_zs = [], []
    pseudo_vals = []
    bg_vals = []
    Z_total = vol_meta.shape[1]

    for t, pos_pts, pos_w, ign_pts in frame_labels:
        vol = vol_meta.frame(t)
        lo, hi = np.percentile(vol, [1, 99.5])
        vol_norm = np.clip((vol.astype(np.float32) - lo) / (hi - lo), 0, 1).astype(np.float32)
        hm = sliding_window_infer(model, vol_norm, patch=TRAIN_CONFIG["patch"], overlap=0.25, device=device)
        bg_vals.append(hm.mean())

        for (z, y, x), w in zip(pos_pts, pos_w):
            z, y, x = int(round(z)), int(round(y)), int(round(x))
            if 0 <= z < hm.shape[0] and 0 <= y < hm.shape[1] and 0 <= x < hm.shape[2]:
                if w >= 0.99:
                    gt_vals.append(hm[z, y, x]); gt_zs.append(z)
                else:
                    pseudo_vals.append(hm[z, y, x])

    print(f"--- {name}  ({len(frame_labels)} frames checked, volume Z={Z_total}) ---")
    if gt_vals:
        print(f"  GT points: n={len(gt_vals)}  mean={np.mean(gt_vals):.3f}  median={np.median(gt_vals):.3f}")
        # split by whether the point sits in the boundary z-region (within 3 slices of either edge)
        edge_vals = [v for v, z in zip(gt_vals, gt_zs) if z <= 2 or z >= Z_total - 3]
        interior_vals = [v for v, z in zip(gt_vals, gt_zs) if not (z <= 2 or z >= Z_total - 3)]
        if edge_vals:
            print(f"    at z-boundary (within 3 slices of edge): n={len(edge_vals)}  mean={np.mean(edge_vals):.3f}")
        if interior_vals:
            print(f"    interior z (away from edge):              n={len(interior_vals)}  mean={np.mean(interior_vals):.3f}")
    else:
        print("  no GT points found in cached frames (unexpected)")
    if pseudo_vals:
        print(f"  pseudo-label points: n={len(pseudo_vals)}  mean={np.mean(pseudo_vals):.3f}  median={np.median(pseudo_vals):.3f}")
    print(f"  background mean (avg across frames): {np.mean(bg_vals):.3f}")
    print()

print("If GT 'interior z' mean is clearly above pseudo-label mean and background, the z=63")
print("boundary case was misleading us -- the model is fine away from volume edges, and the")
print("earlier diagnostic wasn't representative. If GT stays low even away from edges, that's")
print("a real, still-unresolved problem worth investigating further before more training.")


--- 44b6_0113de3b  (20 frames checked, volume Z=64) ---
  GT points: n=20  mean=0.640  median=0.656
    at z-boundary (within 3 slices of edge): n=4  mean=0.484
    interior z (away from edge):              n=16  mean=0.679
  pseudo-label points: n=1461  mean=0.653  median=0.671
  background mean (avg across frames): 0.019

--- 44b6_0b24845f  (20 frames checked, volume Z=64) ---
  GT points: n=20  mean=0.281  median=0.302
    at z-boundary (within 3 slices of edge): n=4  mean=0.269
    interior z (away from edge):              n=16  mean=0.283
  pseudo-label points: n=1912  mean=0.448  median=0.451
  background mean (avg across frames): 0.033

--- 44b6_0c582fdc  (20 frames checked, volume Z=64) ---
  GT points: n=20  mean=0.522  median=0.549
    interior z (away from edge):              n=20  mean=0.522
  pseudo-label points: n=1790  mean=0.550  median=0.560
  background mean (avg across frames): 0.028

If GT 'interior z' mean is clearly above pseudo-label mean and background, the z=63

In [47]:
# ============ EVALUATE: run the trained model through the full pipeline and score it
# with the CORRECTED harness (matches the official metric spec) -- the number directly
# comparable to your classical (73%), hybrid (48.4%), and previous DL (65%) results. ============

def extract_peaks(heatmap, min_distance_vox=(2, 5, 5), threshold=0.3, max_peaks=25000):
    from scipy.ndimage import maximum_filter
    footprint_shape = tuple(2 * d + 1 for d in min_distance_vox)
    mx = maximum_filter(heatmap, size=footprint_shape, mode='nearest')
    peaks_mask = (heatmap == mx) & (heatmap >= threshold)
    coords = np.argwhere(peaks_mask)
    scores = heatmap[peaks_mask]
    if len(coords) > max_peaks:
        idx = np.argsort(scores)[::-1][:max_peaks]
        coords, scores = coords[idx], scores[idx]
    return coords.astype(np.float64), scores.astype(np.float64)

# --- classical tracker (same as your submission notebook, unchanged) ---
class TrackV2:
    __slots__ = ['pos', 'vel', 'score', 'node_id', 'miss', 'alive']
    def __init__(self, pos, score, node_id):
        self.pos = pos.copy(); self.vel = np.zeros(3); self.score = score
        self.node_id = node_id; self.miss = 0; self.alive = True

def link_motion_with_divisions_v2(frames, frame_scores, max_link_um=8.0, motion_weight=0.7,
                                   max_miss=2, appearance_weight=4.0, division_radius_um=6.0,
                                   division_min_gap_um=1.0, division_symmetry_tol=0.6,
                                   max_daughters_per_parent=2):
    node_ids, node_t, node_z, node_y, node_x = [], [], [], [], []
    nid = 1; frame_ids = []; tracks = []
    for t, coords in enumerate(frames):
        ids_t = []; scores_t = frame_scores[t]
        for i, coord in enumerate(coords):
            node_ids.append(nid); node_t.append(t)
            node_z.append(coord[0]); node_y.append(coord[1]); node_x.append(coord[2])
            ids_t.append(nid)
            if t == 0: tracks.append(TrackV2(np.asarray(coord, dtype=np.float64), float(scores_t[i]), nid))
            nid += 1
        frame_ids.append(ids_t)
    def empty_graph():
        return dict(node_t=np.array(node_t), node_z=np.array(node_z), node_y=np.array(node_y),
                     node_x=np.array(node_x), node_ids=np.array(node_ids),
                     edges=np.array([], dtype=np.int64).reshape(-1, 2))
    if len(frames) <= 1: return empty_graph()
    nonempty = [np.asarray(s) for s in frame_scores if len(s)]
    score_scale = np.std(np.concatenate(nonempty)) + 1e-6 if nonempty else 1.0
    edges = []
    for t in range(1, len(frames)):
        cur = np.asarray(frames[t], dtype=np.float64); cur_scores = np.asarray(frame_scores[t], dtype=np.float64)
        current_ids = frame_ids[t]
        alive = [tr for tr in tracks if tr.alive]
        if len(cur) == 0 or len(alive) == 0:
            for tr in alive:
                tr.miss += 1; tr.pos = tr.pos + tr.vel
                if tr.miss > max_miss: tr.alive = False
            continue
        predicted = np.array([tr.pos + tr.vel for tr in alive]); pred_scores = np.array([tr.score for tr in alive])
        d_scaled = ((predicted[:, None, :] - cur[None, :, :]) * SCALE)
        dist = np.sqrt((d_scaled ** 2).sum(axis=2))
        appearance_diff = np.abs(pred_scores[:, None] - cur_scores[None, :]) / score_scale
        cost = np.where(dist <= max_link_um, dist + appearance_weight * appearance_diff, 1e6)
        row_ind, col_ind = linear_sum_assignment(cost)
        matched_track_for_det = {}; used_dets = set()
        for r, c in zip(row_ind, col_ind):
            if dist[r, c] > max_link_um: continue
            tr = alive[r]; edges.append((tr.node_id, current_ids[c]))
            tr.vel = motion_weight * (cur[c] - tr.pos) + (1 - motion_weight) * tr.vel
            tr.pos = cur[c]; tr.score = 0.7 * tr.score + 0.3 * cur_scores[c]
            tr.node_id = current_ids[c]; tr.miss = 0
            matched_track_for_det[c] = r; used_dets.add(c)
        matched_rows = set(matched_track_for_det.values())
        for r, tr in enumerate(alive):
            if r not in matched_rows:
                tr.miss += 1; tr.pos = tr.pos + tr.vel
                if tr.miss > max_miss: tr.alive = False
        unmatched = [c for c in range(len(cur)) if c not in used_dets]
        if unmatched and matched_track_for_det:
            new_tracks = []; daughter_count = defaultdict(lambda: 1)
            for c in unmatched:
                best_r, best_dist = None, None
                for c2, r in matched_track_for_det.items():
                    if daughter_count[r] >= max_daughters_per_parent: continue
                    tr = alive[r]
                    d_parent = np.sqrt((((tr.pos - cur[c]) * SCALE) ** 2).sum())
                    if d_parent > division_radius_um: continue
                    d_primary = dist[r, c2]
                    if d_primary <= 1e-9: continue
                    asym = abs(d_parent - d_primary) / max(d_parent, d_primary)
                    d_between = np.sqrt((((cur[c2] - cur[c]) * SCALE) ** 2).sum())
                    if d_between < division_min_gap_um: continue
                    if asym > division_symmetry_tol: continue
                    if best_dist is None or d_parent < best_dist: best_r, best_dist = r, d_parent
                if best_r is not None:
                    parent_tr = alive[best_r]
                    edges.append((parent_tr.node_id, current_ids[c]))
                    daughter_count[best_r] += 1
                    new_tr = TrackV2(cur[c], cur_scores[c], current_ids[c])
                    new_tr.vel = parent_tr.vel.copy()
                    new_tracks.append(new_tr); used_dets.add(c)
            tracks.extend(new_tracks)
        for c in range(len(cur)):
            if c not in used_dets: tracks.append(TrackV2(cur[c], cur_scores[c], current_ids[c]))
        tracks = [tr for tr in tracks if tr.alive]
    g = empty_graph()
    g["edges"] = np.array(edges, dtype=np.int64).reshape(-1, 2) if edges else np.array([], dtype=np.int64).reshape(-1, 2)
    return g

def close_gaps_enhanced(frames, g, max_gap=1, gap_dist_um=5.0):
    from scipy.ndimage import gaussian_filter1d
    if g.n_edges == 0 or max_gap <= 0: return g
    out_deg = defaultdict(int); in_deg = defaultdict(int)
    for s, t in g.edges: out_deg[s] += 1; in_deg[t] += 1
    node_by_id = {int(nid): i for i, nid in enumerate(g.node_ids)}
    ends = [int(nid) for nid in g.node_ids if out_deg[int(nid)] == 0]
    starts = [int(nid) for nid in g.node_ids if in_deg[int(nid)] == 0]
    new_edges = list(map(tuple, g.edges.tolist()))
    for end_id in ends:
        ei = node_by_id[end_id]; et = int(g.node_t[ei])
        best_s, best_d = None, None
        for start_id in starts:
            si = node_by_id[start_id]; st = int(g.node_t[si])
            gap = st - et
            if 1 <= gap <= max_gap + 1:
                d = np.sqrt((((np.array([g.node_z[ei], g.node_y[ei], g.node_x[ei]]) -
                               np.array([g.node_z[si], g.node_y[si], g.node_x[si]])) * SCALE) ** 2).sum())
                if d <= gap_dist_um * gap and (best_d is None or d < best_d):
                    best_s, best_d = start_id, d
        if best_s is not None:
            new_edges.append((end_id, best_s))
    g.edges = np.array(new_edges, dtype=np.int64).reshape(-1, 2) if new_edges else g.edges
    return g

def prune_isolated(g):
    if g.n_nodes == 0: return g
    connected = set()
    for s, t in g.edges: connected.add(int(s)); connected.add(int(t))
    keep = np.array([int(nid) in connected for nid in g.node_ids])
    if keep.all(): return g
    kept_ids = set(int(nid) for nid in g.node_ids[keep])
    edge_keep = np.array([int(s) in kept_ids and int(t) in kept_ids for s, t in g.edges]) if g.n_edges else np.array([], dtype=bool)
    return TrackGraph(node_t=g.node_t[keep], node_z=g.node_z[keep], node_y=g.node_y[keep],
                       node_x=g.node_x[keep], node_ids=g.node_ids[keep],
                       edges=g.edges[edge_keep] if g.n_edges else g.edges, meta=g.meta)

def smooth_tracks(g, window=3):
    if g.n_edges == 0: return g
    out_edges = defaultdict(list); in_edges = defaultdict(list)
    for s, t in g.edges: out_edges[int(s)].append(int(t)); in_edges[int(t)].append(int(s))
    node_by_id = {int(nid): i for i, nid in enumerate(g.node_ids)}
    starts = [int(nid) for nid in g.node_ids if len(in_edges[int(nid)]) == 0]
    visited = set()
    new_z, new_y, new_x = g.node_z.copy(), g.node_y.copy(), g.node_x.copy()
    for start in starts:
        chain = [start]; cur = start
        while len(out_edges[cur]) == 1:
            nxt = out_edges[cur][0]
            if nxt in visited: break
            chain.append(nxt); cur = nxt
        if len(chain) < 3: continue
        idxs = [node_by_id[c] for c in chain]
        for coord_arr, new_arr in [(g.node_z, new_z), (g.node_y, new_y), (g.node_x, new_x)]:
            vals = coord_arr[idxs]
            k = min(window, len(vals))
            if k < 3: continue
            smoothed = np.convolve(vals, np.ones(k)/k, mode='same')
            for j, idx in enumerate(idxs): new_arr[idx] = smoothed[j]
        visited.update(chain)
    return TrackGraph(node_t=g.node_t, node_z=new_z, node_y=new_y, node_x=new_x,
                       node_ids=g.node_ids, edges=g.edges, meta=g.meta)

def graph_to_rows(name, g):
    rows = []
    for i in range(g.n_nodes):
        rows.append(dict(dataset=name, row_type="node", node_id=int(g.node_ids[i]), t=int(g.node_t[i]),
                          z=int(round(g.node_z[i])), y=int(round(g.node_y[i])), x=int(round(g.node_x[i])),
                          source_id=-1, target_id=-1))
    for s, t in g.edges:
        rows.append(dict(dataset=name, row_type="edge", node_id=-1, t=-1, z=-1, y=-1, x=-1,
                          source_id=int(s), target_id=int(t)))
    return rows

# --- DL detection + tracker, combined ---
def process_dataset_dl_eval(zarr_path, model, device, **kwargs):
    patch = kwargs.get('patch', (32, 128, 128))
    overlap = kwargs.get('overlap', 0.25)
    min_distance_vox = kwargs.get('min_distance_vox', (2, 5, 5))
    threshold = kwargs.get('threshold', 0.5)
    max_peaks = kwargs.get('max_peaks', 25000)
    max_link_um = kwargs.get('max_link_um', 6.5)
    motion_weight = kwargs.get('motion_weight', 0.85)
    max_miss = kwargs.get('max_miss', 1)
    appearance_weight = kwargs.get('appearance_weight', 4.0)
    division_radius_um = kwargs.get('division_radius_um', 5.0)
    division_min_gap_um = kwargs.get('division_min_gap_um', 2.0)
    division_symmetry_tol = kwargs.get('division_symmetry_tol', 0.4)
    max_daughters_per_parent = kwargs.get('max_daughters_per_parent', 2)
    max_gap = kwargs.get('max_gap', 1)
    gap_dist_um = kwargs.get('gap_dist_um', 5.0)

    vol_meta = open_image(zarr_path)
    model.eval()
    frames, frames_scores = [], []
    with torch.no_grad():
        for t in range(vol_meta.n_t):
            vol = vol_meta.frame(t)
            lo, hi = np.percentile(vol, [1, 99.5])
            if hi <= lo: hi = lo + 1.0
            vol_norm = np.clip((vol.astype(np.float32) - lo) / (hi - lo), 0, 1).astype(np.float32)
            hm = sliding_window_infer(model, vol_norm, patch=patch, overlap=overlap, device=device)
            coords, scores = extract_peaks(hm, min_distance_vox=min_distance_vox, threshold=threshold, max_peaks=max_peaks)
            frames.append(coords); frames_scores.append(scores)
            del vol
            if t % 10 == 0: gc.collect()

    g_dict = link_motion_with_divisions_v2(
        frames, frames_scores, max_link_um=max_link_um, motion_weight=motion_weight, max_miss=max_miss,
        appearance_weight=appearance_weight, division_radius_um=division_radius_um,
        division_min_gap_um=division_min_gap_um, division_symmetry_tol=division_symmetry_tol,
        max_daughters_per_parent=max_daughters_per_parent)
    g = TrackGraph(**g_dict, meta={})
    g = close_gaps_enhanced(frames, g, max_gap=max_gap, gap_dist_um=gap_dist_um)
    g = prune_isolated(g)
    g = smooth_tracks(g)
    return g

# --- corrected harness (matches the official metrics.md) ---
def get_estimated_node_count(geff_path):
    import zarr
    z = zarr.open(geff_path, mode="r")
    return int(z.attrs.get("estimated_number_of_nodes", -1))

def match_nodes(pred, gt, max_dist_um=7.0):
    pred_id_to_gt, gt_id_to_pred = {}, {}
    frames_t = sorted(set(np.asarray(gt["t"]).tolist()))
    for tp in frames_t:
        p_idx = np.where(np.asarray(pred["t"]) == tp)[0]
        g_idx = np.where(np.asarray(gt["t"]) == tp)[0]
        if len(p_idx) == 0 or len(g_idx) == 0: continue
        p_xyz = np.stack([np.asarray(pred["z"])[p_idx], np.asarray(pred["y"])[p_idx],
                           np.asarray(pred["x"])[p_idx]], axis=1) * SCALE
        g_xyz = np.stack([np.asarray(gt["z"])[g_idx], np.asarray(gt["y"])[g_idx],
                           np.asarray(gt["x"])[g_idx]], axis=1) * SCALE
        d = np.sqrt((((p_xyz[:, None, :] - g_xyz[None, :, :]) ** 2)).sum(axis=2))
        cost = np.where(d <= max_dist_um, d, 1e6)
        r, c = linear_sum_assignment(cost)
        for ri, ci in zip(r, c):
            if d[ri, ci] <= max_dist_um:
                pid = int(np.asarray(pred["node_ids"])[p_idx[ri]]); gid = int(np.asarray(gt["node_ids"])[g_idx[ci]])
                pred_id_to_gt[pid] = gid; gt_id_to_pred[gid] = pid
    return pred_id_to_gt, gt_id_to_pred

def edge_jaccard_correct(pred, gt, max_dist_um=7.0):
    pred_id_to_gt, gt_id_to_pred = match_nodes(pred, gt, max_dist_um)
    gt_edges = set(map(tuple, np.asarray(gt["edges"]).tolist()))
    pred_edges = set(map(tuple, np.asarray(pred["edges"]).tolist()))
    gt_source_of_target = defaultdict(set); gt_targets_of_source = defaultdict(set)
    for (gs, gtid) in gt_edges:
        gt_source_of_target[gtid].add(gs); gt_targets_of_source[gs].add(gtid)
    tp = 0; fp = 0; matched_gt_edges = set()
    for (s, t) in pred_edges:
        gs = pred_id_to_gt.get(int(s)); gtid = pred_id_to_gt.get(int(t))
        if gs is not None and gtid is not None and (gs, gtid) in gt_edges:
            tp += 1; matched_gt_edges.add((gs, gtid)); continue
        is_fp = False
        if gtid is not None and len(gt_source_of_target[gtid]) > 0 and gs not in gt_source_of_target[gtid]: is_fp = True
        if gs is not None and len(gt_targets_of_source[gs]) > 0 and gtid not in gt_targets_of_source[gs]: is_fp = True
        if is_fp: fp += 1
    fn = len(gt_edges) - len(matched_gt_edges)
    denom = tp + fp + fn
    jac = tp / denom if denom > 0 else 0.0
    return dict(tp=tp, fp=fp, fn=fn, edge_jaccard=jac, n_pred_nodes=len(pred["node_ids"]), n_matched_nodes=len(pred_id_to_gt))

def adjusted_edge_jaccard(pred, gt, t_true, max_dist_um=7.0, a=0.1):
    res = edge_jaccard_correct(pred, gt, max_dist_um)
    t_pred = res["n_pred_nodes"]
    adj = max(0.0, res["edge_jaccard"] * (1 - a * (t_pred - t_true) / t_true)) if t_true and t_true > 0 else res["edge_jaccard"]
    res["t_pred"] = t_pred; res["t_true"] = t_true; res["adjusted_jaccard"] = adj
    return res

def evaluate_dl_on_train(train_dir, dataset_names, model, device, infer_config, max_dist_um=7.0):
    results = []
    for name in dataset_names:
        zarr_path = os.path.join(train_dir, name + ".zarr")
        geff_path = os.path.join(train_dir, name + ".geff")
        if not (os.path.isdir(zarr_path) and os.path.isdir(geff_path)):
            print(f"  skipping {name}: missing .zarr or .geff"); continue
        gt = load_geff(geff_path)
        t_true = get_estimated_node_count(geff_path)
        g = process_dataset_dl_eval(zarr_path, model, device, **infer_config)
        pred = dict(node_ids=g.node_ids, t=g.node_t, z=g.node_z, y=g.node_y, x=g.node_x, edges=g.edges)
        res = adjusted_edge_jaccard(pred, gt, t_true, max_dist_um=max_dist_um)
        print(f"  {name}: tp={res['tp']} fp={res['fp']} fn={res['fn']}  edge_jaccard={res['edge_jaccard']:.4f}  "
              f"adjusted={res['adjusted_jaccard']:.4f}  (T_pred={res['t_pred']}, T_true={res['t_true']}, matched={res['n_matched_nodes']})")
        results.append(res)
    if not results: return dict(mean_edge_jaccard=0.0, mean_adjusted_jaccard=0.0)
    return dict(mean_edge_jaccard=float(np.mean([r['edge_jaccard'] for r in results])),
                mean_adjusted_jaccard=float(np.mean([r['adjusted_jaccard'] for r in results])))

# ---- run it on the same 3 samples used for your classical/hybrid evaluations ----
EVAL_SAMPLES = ["44b6_0113de3b", "44b6_0b24845f", "44b6_0c582fdc"]  # adjust if you used different ones
INFER_EVAL_CONFIG = dict(
    patch=TRAIN_CONFIG["patch"], overlap=0.25, min_distance_vox=(2, 5, 5),
    threshold=0.5, max_peaks=25000,
    max_link_um=6.5, motion_weight=0.85, max_miss=1, appearance_weight=4.0,
    division_radius_um=5.0, division_min_gap_um=2.0, division_symmetry_tol=0.4, max_daughters_per_parent=2,
    max_gap=1, gap_dist_um=5.0,
)

print("Evaluating trained model on:", EVAL_SAMPLES)
dl_result = evaluate_dl_on_train(TRAIN_DIR, EVAL_SAMPLES, model, device, INFER_EVAL_CONFIG)
print(f"\nmean_edge_jaccard={dl_result['mean_edge_jaccard']:.4f}  mean_adjusted_jaccard={dl_result['mean_adjusted_jaccard']:.4f}")
print("Compare mean_adjusted_jaccard directly against your classical (~0.63 local / 73% leaderboard)")
print("and hybrid (48.4% leaderboard) results.")


Evaluating trained model on: ['44b6_0113de3b', '44b6_0b24845f', '44b6_0c582fdc']
  44b6_0113de3b: tp=42 fp=9 fn=8  edge_jaccard=0.7119  adjusted=0.7119  (T_pred=18412, T_true=-1, matched=47)
  44b6_0b24845f: tp=1 fp=3 fn=48  edge_jaccard=0.0192  adjusted=0.0192  (T_pred=7764, T_true=-1, matched=3)
  44b6_0c582fdc: tp=26 fp=20 fn=44  edge_jaccard=0.2889  adjusted=0.2889  (T_pred=14755, T_true=-1, matched=37)

mean_edge_jaccard=0.3400  mean_adjusted_jaccard=0.3400
Compare mean_adjusted_jaccard directly against your classical (~0.63 local / 73% leaderboard)
and hybrid (48.4% leaderboard) results.
